# Figure 4 — pulse and chirp impacts of perturbations

Run cells in order. Calculated panels are saved under `figures/`.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "notebooks").is_dir() and (p / "data").is_dir())
sys.path.insert(0, str(ROOT / "src"))
FIGURES = ROOT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
SAVED_RESULTS = ROOT / "saved_results"

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
plt.style.use(ROOT / "notebooks/ncb.mplstyle")
plt.close("all")
np.random.seed(0)


## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import plotting as p

from metpy.interpolate import interpolate_to_points
import jax.numpy as jnp

from Gbim_import_dat import get_Gbim_data_chirpfield, get_Gbim_data_DC
from stim_gen import makeCHIRPstim, makePPFstim

MAX = 20
step = 5
cpauMax = 1000 + MAX
max_len = cpauMax

## Load behavior

In [ ]:
[CDUR, CPAU, CPER, DCs, NPUL, rXY]= get_Gbim_data_chirpfield()
[CDUR_dc, CPAU_dc, CPER_dc, DCs_dc, NPUL_dc, rXY_dc]=get_Gbim_data_DC()

keys = ['CDUR', 'CPAU', 'CPER', 'CDC', 'NPUL', 'rXY']
data_cpf = dict(zip(keys, [CDUR, CPAU, CPER, DCs, NPUL, rXY]))
data_dc = dict(zip(keys, [CDUR_dc, CPAU_dc, CPER_dc, DCs_dc, NPUL_dc, rXY_dc]))

data_all = {key: np.concatenate((data_cpf[key], data_dc[key])) for key in keys}

## Interpolate behavior

In [ ]:
duri0 = paui0 = np.arange(0, max_len, step)
duri, paui = np.meshgrid(duri0,paui0)

CDURinterp=np.concatenate((CDUR,[0, 0]))
CPAUinterp=np.concatenate((CPAU,[0, 1000]))
rXYinterp=np.concatenate((rXY,[0, 0]))

points = np.array([CDURinterp + np.random.randn(*CDURinterp.shape)/1_000, CPAUinterp+ np.random.randn(*CPAUinterp.shape)/1_000]).T
new_points = np.array([duri, paui]).T.reshape((-1, 2))

cpf_behave = interpolate_to_points(points, np.clip(rXYinterp, 0, np.inf), new_points, interp_type='natural_neighbor', minimum_neighbors=4, rbf_smooth=0)
cpf_behave = np.maximum(cpf_behave, 0)
cpf_behave = cpf_behave.reshape((len(duri), len(paui))).T

## Compute or load pulse and chirp fields

In [ ]:
RUN_EXPENSIVE = False  # Set True to recompute parameter grids and refresh the saved fields.
CACHE = (ROOT / "results" if RUN_EXPENSIVE else SAVED_RESULTS) / "fig4_fields.npz"

if RUN_EXPENSIVE:
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    input_stim, s, cpau, cdur = makeCHIRPstim(
        cpauMax=1000 + MAX,
        cdurMax=1000 + MAX,
        cpauMin=MAX,
        cdurMin=MAX,
        ppau=20,
        pdur=20,
        Fs=1000,
        num_steps=32,
        stim_len=10_000,
    )
    input_stim = jnp.array(input_stim)
    points = np.array(
        [
            s["cpau"] + np.random.randn(*s["cpau"].shape) / 100,
            s["cdur"] + np.random.randn(*s["cdur"].shape) / 100,
        ]
    ).T

    xi0 = yi0 = np.arange(0, cpauMax, step)
    xi, yi = np.meshgrid(xi0, yi0)
    new_points = np.array([xi, yi]).T.reshape((-1, 2))

    cduri = xi.copy()
    cpaui = yi.copy()

    from models.fix import Params, Params_noada, Params_noada_AN1, Params_noada_LN3, Params_noInh_from_LN2, Params_noInh_in_AN1, Params_noInh_in_LN5, Params_noInh_AN1_LN4, Params_noInh, Params_noInh_noada

    params = {'full': Params(),
              'noboth': Params_noInh_noada(),
              'noada': Params_noada(), 'noada_AN1': Params_noada_AN1(), 'noada_LN3': Params_noada_LN3(),
              'noinh': Params_noInh(), 'noinh_LN4': Params_noInh_from_LN2(), 'noInh_AN1': Params_noInh_in_AN1(), 'noInh_LN5': Params_noInh_in_LN5(), 'noInh_AN1_LN4': Params_noInh_AN1_LN4()}
    resp_ln4 = {}
    model_outputs = {}
    cpf = {}

    for key, param in params.items():
        print(key)
        model_output = run_model(input_stim, param, verbose_output=False, return_dict=True)
        model_outputs[key] = model_output
        resp_ln4[key] = np.nanmean(model_output['ln4'], axis=0)
        cpf[key] = interpolate_to_points(points, resp_ln4[key], new_points, interp_type='rbf', rbf_func='linear', rbf_smooth=20)#, minimum_neighbors=1)
        cpf[key] = cpf[key].reshape((len(xi), len(yi)))

    resp_ln4['stim'] = np.mean(input_stim, axis=0)
    cpf['stim'] = interpolate_to_points(points, resp_ln4['stim'], new_points, interp_type='rbf', rbf_func='linear', rbf_smooth=20)#, minimum_neighbors=1)
    cpf['stim'] = cpf['stim'].reshape((len(xi), len(yi)))

    cpf['behavior'] = cpf_behave

    s['rXY'] = resp_ln4['full']
    for key, resp in resp_ln4.items():
        s[f'rXY_{key}'] = resp

    stim_matrix, stim_dict, ppau, pdur = makePPFstim(ppauMax=80, pdurMax=80, ppauMin=0, pdurMin=0, cdur=200, cpau=200, Fs=1000, step=1, ppau=None, pdur=None)

    ppf = {}
    for key, param in params.items():
        model_output = run_model(stim_matrix, param, verbose_output=True, return_dict=True)
        ppf[key] = np.nanmean(model_output['ln4'], axis=0).reshape((80,80))
    names = ["full", "noada_AN1", "noInh_AN1", "noInh_LN5", "noada_LN3", "noinh_LN4"]
    np.savez_compressed(CACHE, keys=names,
                        pulse=np.array([ppf[k] for k in names]),
                        chirp=np.array([cpf[k] for k in names]))
else:
    if not CACHE.exists():
        raise FileNotFoundError(f"{CACHE} is missing. Set RUN_EXPENSIVE = True and run this cell once.")
    with np.load(CACHE) as cached:
        names = cached["keys"].tolist()
        ppf = dict(zip(names, cached["pulse"]))
        cpf = dict(zip(names, cached["chirp"]))
    ppau = pdur = np.arange(80)

## Compare pulse and chirp impacts

In [ ]:
keys = ['full', 'noada_AN1', 'noInh_AN1', 'noInh_LN5', 'noada_LN3', 'noinh_LN4']
labels = {'full': 'Full model', 'noada_AN1': 'No div. norm. in AN1',
          'noInh_AN1': 'No. inh. comp. in AN1', 'noInh_LN5': 'No. inhibition in LN5',
          'noada_LN3': 'No div. norm in LN3', 'noinh_LN4': 'No inhibition in LN4'}

plt.figure(figsize=(16, 10))

for cnt, key in enumerate(keys):
    ax = plt.subplot(3, len(keys), cnt+1)
    im = plt.pcolormesh(pdur, ppau, ppf[key]/np.max(ppf[key]), cmap='Greens', rasterized=True)
    plt.plot([0, 30], [30, 0], c='k')
    plt.plot([0, 40], [40, 0], c='k')
    plt.axis('square')
    plt.xlim(0, 80)
    plt.ylim(0, 80)
    plt.xticks([0, 80])
    plt.yticks([0, 80])
    plt.clim(0, 1)
    plt.title(labels[key].replace(' in ', '\nin '), fontsize=11)
    if cnt==0:
        plt.xlabel('Pulse duration [ms]')
        plt.ylabel('Pulse pause [ms]')
    else:
        plt.yticks([0, 1], labels=[])

    if cnt==len(keys)-1:
        cax = inset_axes(
                ax,
                width="7%",
                height="35%",
                loc="upper right",
            )
        plt.colorbar(im, cax=cax)

for cnt, key in enumerate(keys):
    ax = plt.subplot(3, len(keys), cnt+1 + len(keys))
    im = plt.pcolormesh(duri/1000, paui/1000, cpf[key] / np.max(cpf[key]), cmap='Greys', rasterized=True)
    plt.axis('square')
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.xticks([0, 1])
    plt.yticks([0, 1])
    plt.clim(0, 1)
    plt.plot([0, 1], [0, .25], c='seagreen')
    if cnt==0:
        plt.xlabel('Chirp duration [s]')
        plt.ylabel('Chirp pause [s]')
    else:
        plt.yticks([0, 1], labels=[])

    if cnt==len(keys)-1:
        cax = inset_axes(
                ax,
                width="7%",
                height="35%",
                loc="upper right",
            )
        plt.colorbar(im, cax=cax)

impact_keys = keys[1:]
d_ppf, d_cpf = [], []
for cnt, key in enumerate(keys[1:]):
    d_ppf.append(np.sqrt(np.mean((p.nmax(ppf['full']) - p.nmax(ppf[key]))**2)))
    d_cpf.append(np.sqrt(np.mean((p.nmax(cpf['full']) - p.nmax(cpf[key]))**2)))

plt.subplot(3,3,7)
plt.plot(d_ppf, 'o-g', label='Pulse', markersize=10)
plt.plot(d_cpf, 'o-k', label='Chirp', markersize=10)
plt.xlim(-0.5, 4.5)
plt.xticks(range(5), labels=['AN1 norm.', 'AN1 filter', 'LN5', 'LN3', 'LN4'], rotation=30)
plt.yticks([0, 0.3])
plt.ylabel('$\Delta$ tuning')
plt.ylim(0)
plt.legend(handlelength=0, handletextpad=0,
           markerscale=0, labelcolor='linecolor')

plt.subplot(3,3,8)
for x, y, key in zip(d_ppf, d_cpf, keys[1:]):
    plt.plot(x, y, 'o', label=labels[key], markersize=10)

plt.plot([0, 0.4], [0, 0.4], '-k')
plt.xticks([0, 0.2, 0.4])
plt.yticks([0, 0.2, 0.4])
plt.axis('square')
plt.xlabel('$\Delta$ pulse tuning')
plt.ylabel('$\Delta$ chirp tuning')
plt.legend(handlelength=0, handletextpad=0,
           markerscale=0, labelcolor='linecolor')

p.label_axes(size=12, loc=(0.02, 0.95))
plt.savefig(f'{FIGURES}/fig_4.pdf', bbox_inches='tight')